# Module 16B - Synthetic data

Use this notebook after `tests/test_synth.py` is passing, with your Module 13 dataset on disk and ProdLM running. You'll point the teacher model at your fifty hand-authored pairs, generate ~150 synthetic ones through your own filters, audit the result, and settle the LIMA question on your own model: does more, noisier data beat less, cleaner data? The referee is always your held-out hand data.

The important work is reading the funnel and the outputs like an editor. Generation is two backend calls; curation is the product.

1. Read the lesson page (`docs/modules/16b-synthetic-data.md`).
2. Open this notebook with `./notebook.sh 16b`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

## Setup

In [ ]:
from pathlib import Path
import json
import random
import subprocess
import sys

import torch

from g2c.artifacts import load_model_artifact_with_tokenizer
from g2c.inference import load_selected_backend
from g2c.notebook_extras.sft import chat_sample, train_sft_with_progress
from g2c.notebook_extras.synth import show_pair_sample, synthesize_with_progress
from g2c.sft import ChatTemplate, SFTTrainer
from g2c.synth import (
    dedupe_pairs,
    ngram_overlap,
    propose_instructions,
    validate_pair,
)

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

Run the Module 16B tests before proceeding. In the clean scaffold this cell should fail until you implement `ngram_overlap` and `dedupe_pairs`.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_synth.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 16B synth tests are not passing yet."

## Load and split your Module 13 dataset

One split, two jobs, one rule. The **training side** seeds the factory and joins the fine-tunes; the **held-out side** judges every fine-tune and *never* touches generation. Seeding from the validation split would leak near-rephrasings of your eval items into training — the split discipline below prevents it by construction.

(The split uses Module 13's seed, so your held-out set here is the same one your Module 13 run validated against.)

In [ ]:
dataset_path = repo_root / "data" / "work" / "module13" / "instructions.json"
if not dataset_path.exists():
    raise FileNotFoundError(
        f"{dataset_path} not found -- run Module 13's dataset cells first "
        "(./notebook.sh 13). This module seeds generation from that dataset."
    )
hand_pairs = json.loads(dataset_path.read_text())

DATA_SPLIT_SEED = 13  # same split as Module 13: same held-out referee
perm = torch.randperm(
    len(hand_pairs), generator=torch.Generator().manual_seed(DATA_SPLIT_SEED)
).tolist()
val_count = max(1, len(hand_pairs) // 5)
val_indices = set(perm[:val_count])
hand_train = [p for i, p in enumerate(hand_pairs) if i not in val_indices]
hand_val = [p for i, p in enumerate(hand_pairs) if i in val_indices]
print(f"hand-authored: {len(hand_train)} train (seed the factory) / {len(hand_val)} held out (the referee)")

## The teacher

`ProdLM` through Module 16's `Backend` interface. Any backend with `complete()` works, but the point of a teacher is that it's stronger than the student.

In [ ]:
MODEL_SELECTION = "ProdLM"
PRODLM_MODEL_ID = None  # optional Ollama tag override

teacher = None
try:
    teacher = load_selected_backend(
        MODEL_SELECTION,
        repo_root=repo_root,
        prodlm_model_id=PRODLM_MODEL_ID,
        required=False,
    )
    if teacher is None:
        print("No teacher backend available. Run ./prodlm.sh and re-run this cell.")
    else:
        print("teacher:", teacher.info)
except Exception as exc:
    print(f"Teacher unavailable: {type(exc).__name__}: {exc}")

## Exercise 1 - Read one batch

One live proposal round: the teacher sees six of your seed instructions and is asked for eight new ones. Read them the way an editor would, before any filter touches them.

In [ ]:
assert teacher is not None, "start ProdLM first (./prodlm.sh)"

seed_instructions = [p["user"] for p in hand_train]
rng = random.Random(0)
first_batch = propose_instructions(teacher, seed_instructions, count=8, k=6, rng=rng)
for i, instruction in enumerate(first_batch, 1):
    print(f"{i}. {instruction}")

In [ ]:
"Question: Read the eight proposals as an editor. How many are genuinely new ideas versus recombinations of your seeds? Are any unanswerable in one or two sentences, or unanswerable by a small model at all? Quote one good one and one you'd reject."
"Answer: "

## Exercise 2 - Measure mode collapse

Ask for enough batches and the teacher starts repeating itself. Your `ngram_overlap` makes that a number: for each raw proposal, its maximum overlap against everything proposed before it (plus your seeds). Watch the duplicate rate climb batch over batch.

In [ ]:
RAW_BATCHES = 4
DUP_THRESHOLD = 0.5

pool = list(seed_instructions)
per_batch_rates = []
raw_rng = random.Random(1)
for b in range(RAW_BATCHES):
    batch = propose_instructions(teacher, seed_instructions, count=8, k=6, rng=raw_rng)
    dups = 0
    for candidate in batch:
        if any(ngram_overlap(candidate, seen) >= DUP_THRESHOLD for seen in pool):
            dups += 1
        pool.append(candidate)
    rate = dups / max(1, len(batch))
    per_batch_rates.append(rate)
    print(f"batch {b + 1}: {len(batch)} proposed, {dups} near-duplicates ({rate:.0%})")

print(f"\noverall duplicate rate at threshold {DUP_THRESHOLD}: "
      f"{sum(per_batch_rates) / len(per_batch_rates):.0%}")

In [ ]:
"Question: Report your per-batch duplicate rates. Did the rate climb as the pool grew, and why would it? The proposal call re-samples which few-shot seeds the teacher sees each round -- what would you expect these numbers to do if the examples were fixed instead?"
"Answer: "

## Exercise 3 - Run the factory

The full Self-Instruct loop: propose hot, gate, dedup against the growing pool, answer cool, gate again. The funnel table at the end is the deliverable — data quality, as numbers. Expect this to take 10–25 minutes for 150 pairs; the marginal pair gets slower as the duplicate rate climbs.

In [ ]:
TARGET_PAIRS = 150
OVERLAP_THRESHOLD = 0.7

synthetic_pairs, funnel = synthesize_with_progress(
    teacher,
    hand_train,
    target=TARGET_PAIRS,
    batch_count=8,
    threshold=OVERLAP_THRESHOLD,
    seed=16,
)

out_dir = repo_root / "data" / "work" / "module16b"
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "synthetic-instructions.json").write_text(
    json.dumps(synthetic_pairs, indent=2) + "\n"
)
(out_dir / "funnel.json").write_text(json.dumps(funnel, indent=2) + "\n")
print(f"saved {len(synthetic_pairs)} pairs and the funnel to {out_dir.relative_to(repo_root)}")

## Exercise 4 - Audit ten pairs

Filters catch shape, not sense. Hand-grade a reproducible random sample: is the instruction reasonable, is the answer *correct*, would you pay a labeling vendor who delivered this?

In [ ]:
show_pair_sample(synthetic_pairs, k=10, seed=4)

In [ ]:
"Question: Grade your ten sampled pairs: how many have factually or logically wrong answers? Name the failure modes you saw (wrong facts, hedging, style drift, question-as-answer...). Estimate the dataset's error rate, and make the call: would you accept this batch from a data vendor?"
"Answer: "

## Exercise 5 - The three-way fine-tune

Hand vs. synthetic vs. mixed — same base model, same trainer, same step budget, and the same referee: masked val loss on your held-out *hand-authored* pairs. Runs are sequential with one model in memory at a time; budget roughly 3× your Module 13 run, or drop `max_steps` for a faster first pass.

In [ ]:
template = ChatTemplate()
TRAIN_DEVICE = "auto"
SEED = 16
PROBE_PROMPTS = [
    "What is the capital of France?",
    "Give one tip for training small models.",
    "What does the loss mask do in SFT?",
]

SFT_CONFIG = {
    "max_seq_len": 128,
    "batch_size": 4,
    "max_steps": 300,
    "max_lr": 3e-4,  # BaseLM's SFT lr, as in Module 13
    "min_lr": 3e-5,
    "warmup_steps": 20,
    "weight_decay": 0.01,
    "grad_clip": 1.0,
    "eval_every": 50,
    "eval_iters": 10,
    "log_every": 10,
    "device": TRAIN_DEVICE,
}

def messages_from_pair(pair):
    return [
        {"role": "user", "content": pair["user"]},
        {"role": "assistant", "content": pair["assistant"]},
    ]

datasets = {
    "hand": hand_train,
    "synthetic": synthetic_pairs,
    "mixed": hand_train + synthetic_pairs,
}
results = {}
for name, pairs in datasets.items():
    artifact = load_model_artifact_with_tokenizer(
        "BaseLM", repo_root=repo_root, device=TRAIN_DEVICE
    )
    tokenizer = artifact.tokenizer
    pad_id = tokenizer.special_to_id.get(
        "<|pad|>", getattr(tokenizer, "pad_token_id", None) or 0
    )
    encode = lambda ps: [
        template.render_with_mask(
            messages_from_pair(p), tokenizer, vocab_size=artifact.model.vocab_size
        )
        for p in ps
    ]
    trainer = SFTTrainer(
        artifact.model,
        examples=encode(pairs),
        generator=torch.Generator().manual_seed(SEED),
        pad_id=pad_id,
        **SFT_CONFIG,
    )
    history = train_sft_with_progress(
        f"SFT on {name} ({len(pairs)} pairs)", trainer, eval_examples=encode(hand_val)
    )
    results[name] = {
        "pairs": len(pairs),
        "val_loss": history["val_loss"][-1] if history["val_loss"] else None,
        "samples": {p: chat_sample(artifact, p, seed=SEED) for p in PROBE_PROMPTS},
    }
    del artifact, trainer  # keep one BaseLM in memory at a time

print(f"{'dataset':<12}{'pairs':>8}{'hand-val loss':>16}")
for name, r in results.items():
    print(f"{name:<12}{r['pairs']:>8}{r['val_loss']:>16.4f}")

In [ ]:
for prompt in PROBE_PROMPTS:
    print("=" * 70)
    print("PROMPT:", prompt)
    for name in results:
        sample = results[name]["samples"][prompt].strip().replace("\n", " ")
        print(f"  {name:<10} {sample[:120]}")

In [ ]:
"Question: Read the table and the sample sweep. Which dataset won on held-out hand-val loss, and does the sample quality agree? Did ~3x synthetic data beat your 40 hand pairs -- and what does the mixed run tell you about whether the two data sources are redundant or complementary?"
"Answer: "

## Exercise 6 (optional) - Style imprinting

Your hand data has your voice. The synthetic data has the teacher's. Look for the teacher's fingerprints in the synthetic-trained model — phrasing that appears nowhere in your forty pairs.

In [ ]:
openers = ("sure", "certainly", "of course", "great question", "as an ai")
print(f"{'dataset':<12}{'mean answer chars':>20}{'teacher-ish openers':>22}")
for name, r in results.items():
    samples = list(r["samples"].values())
    mean_len = sum(len(s) for s in samples) / len(samples)
    openings = sum(s.strip().casefold().startswith(openers) for s in samples)
    print(f"{name:<12}{mean_len:>20.0f}{openings:>22}")

hand_lengths = sum(len(p["assistant"]) for p in hand_train) / len(hand_train)
synth_lengths = sum(len(p["assistant"]) for p in synthetic_pairs) / len(synthetic_pairs)
print(f"\ntraining-data answer length: hand {hand_lengths:.0f} chars, synthetic {synth_lengths:.0f} chars")

In [ ]:
"Question: Where do you see the teacher's fingerprints? Cite one phrasing or formatting habit in the synthetic-trained model's outputs that your hand data never contained, and trace it back to a synthetic training pair if you can."
"Answer: "

## You just distilled a model

What you ran today has a name. "Distillation" means two things: the classic sense (Hinton 2015 — train the student on the teacher's temperature-softened *logits*) and the sequence-level sense (Kim & Rush 2016 — train the student on teacher-*written text* as hard labels). The second is what Alpaca did, what "distilled from GPT-4" means on a model card, and what you just did to ProdLM.

When complete, ask a coding agent to grade your Module 16B notebook. Partial work is fine: the agent should grade what's answered and skip blank answers.

In [ ]:
"Question: In both senses of the word: which kind of distillation did you perform today? For the kind you could NOT perform, name the exact missing ingredient in this stack -- there are two independent blockers; give at least one, and say where in this course you met the same limitation before."
"Answer: "